# Lab Assignment 2 - Part C: Naive Bayes for Spam Detection
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement a **Naive Bayes classifier** for spam email detection using the Spambase dataset.

### Dataset Description
The Spambase dataset contains 4601 email samples with 57 features:
- **Features 1-48**: Word frequencies (percentage of words matching specific words)
- **Features 49-54**: Character frequencies (`;`, `(`, `[`, `!`, `$`, `#`)
- **Features 55-57**: Capital letter statistics
- **Label**: 1 = spam, 0 = not spam

### Your Tasks
1. **Implement Gaussian Naive Bayes** from scratch
2. **Train and evaluate** your classifier (accuracy, precision, recall, F1-score)
3. **Feature analysis**: Identify top discriminative features
4. **Discussion**: Why does Naive Bayes work for spam detection?


## Setup


In [ ]:
%pip install ucimlrepo


In [ ]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo


## Load the Spambase Dataset


In [ ]:
# Fetch Spambase dataset from UCI ML Repository
spambase = fetch_ucirepo(id=94)

# Get features and labels
X = spambase.data.features.values
y = spambase.data.targets.values.ravel()

print(f"Dataset shape: {X.shape}")
print(f"Number of spam emails: {np.sum(y == 1)}")
print(f"Number of non-spam emails: {np.sum(y == 0)}")
print(f"\nFeature names: {list(spambase.data.features.columns[:10])}...")  # First 10 features


In [ ]:
# Split data into training (80%) and testing (20%) sets
def train_test_split(X, y, test_size=0.2, random_state=42):
    """
    Split data into training and testing sets.
    """
    np.random.seed(random_state)
    n_samples = len(y)
    indices = np.random.permutation(n_samples)
    test_size = int(n_samples * test_size)
    
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]
    
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")


## Task 1: Implement Gaussian Naive Bayes
Implement a Gaussian Naive Bayes classifier from scratch.

**Key formulas:**
- Class prior: $P(C) = \frac{N_C}{N}$
- Gaussian likelihood: $P(x_i|C) = \frac{1}{\sqrt{2\pi\sigma_{i,C}^2}} \exp\left(-\frac{(x_i - \mu_{i,C})^2}{2\sigma_{i,C}^2}\right)$
- Use **log-probabilities** to avoid numerical underflow!


In [ ]:
class GaussianNaiveBayes:
    """
    Gaussian Naive Bayes classifier from scratch.
    Uses log-probabilities to avoid numerical underflow.
    """

    def __init__(self, var_smoothing=1e-9):
        self.classes      = None
        self.priors       = None   # P(C) for each class
        self.means        = None   # μ_{i,C}: mean of feature i for class C
        self.variances    = None   # σ²_{i,C}: variance of feature i for class C
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        """
        Estimate class priors and per-class Gaussian parameters.

        Parameters:
        -----------
        X : numpy array (n_samples, n_features)
        y : numpy array (n_samples,)
        """
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes    = len(self.classes)

        self.priors    = np.zeros(n_classes)
        self.means     = np.zeros((n_classes, n_features))
        self.variances = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.priors[i]    = X_c.shape[0] / n_samples          # P(C) = N_C / N
            self.means[i]     = X_c.mean(axis=0)                   # μ_{i,C}
            self.variances[i] = X_c.var(axis=0) + self.var_smoothing  # σ²_{i,C}

    def _gaussian_log_likelihood(self, X, mean, var):
        """
        Log of Gaussian PDF for all samples and features:
        log P(x_i | C) = -0.5 * log(2π·σ²) - 0.5 * (x - μ)² / σ²

        Parameters:
        -----------
        X    : (n_samples, n_features)
        mean : (n_features,)
        var  : (n_features,)

        Returns:
        --------
        log_likelihood : (n_samples, n_features)
        """
        return -0.5 * np.log(2 * np.pi * var) - 0.5 * ((X - mean) ** 2) / var

    def _class_log_posteriors(self, X):
        """Compute log posterior for each class for all samples."""
        n_samples = X.shape[0]
        n_classes = len(self.classes)
        log_posts = np.zeros((n_samples, n_classes))

        for i in range(n_classes):
            log_prior      = np.log(self.priors[i])
            log_likelihood = self._gaussian_log_likelihood(X, self.means[i], self.variances[i])
            log_posts[:, i] = log_prior + log_likelihood.sum(axis=1)

        return log_posts

    def predict(self, X):
        """
        Predict class labels (class with highest log posterior).

        Parameters:
        -----------
        X : numpy array (n_samples, n_features)

        Returns:
        --------
        predictions : numpy array (n_samples,)
        """
        log_posts = self._class_log_posteriors(X)
        return self.classes[np.argmax(log_posts, axis=1)]

    def predict_proba(self, X):
        """
        Return normalized probability estimates via softmax of log posteriors.

        Returns:
        --------
        probabilities : numpy array (n_samples, n_classes)
        """
        log_posts = self._class_log_posteriors(X)
        log_posts -= log_posts.max(axis=1, keepdims=True)  # numerical stability
        probs = np.exp(log_posts)
        probs /= probs.sum(axis=1, keepdims=True)
        return probs


print("GaussianNaiveBayes class defined.")


## Task 2: Train and Evaluate
Train your classifier and compute evaluation metrics.


In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Compute accuracy, precision, recall, and F1-score.
    Assumes binary classification with positive class = 1.

    Returns:
    --------
    accuracy, precision, recall, f1_score : floats
    """
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    return float(accuracy), float(precision), float(recall), float(f1)


def confusion_matrix(y_true, y_pred):
    """
    Build 2x2 confusion matrix:
        [[TN, FP],
         [FN, TP]]

    Returns:
    --------
    matrix : numpy array of shape (2, 2)
    """
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    return np.array([[TN, FP], [FN, TP]])


print("Defined: compute_metrics, confusion_matrix")


## Task 3: Feature Analysis
Identify the most discriminative features for spam detection.


In [ ]:
# Train Gaussian Naive Bayes classifier
nb_classifier = GaussianNaiveBayes()
nb_classifier.fit(X_train, y_train)

# Predict on test set
y_pred = nb_classifier.predict(X_test)

# Compute and print metrics
accuracy, precision, recall, f1 = compute_metrics(y_test, y_pred)
print("=" * 42)
print("  Gaussian Naive Bayes — Test Results")
print("=" * 42)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix (rows=Actual, cols=Predicted):")
print(f"                  Pred:Not-Spam  Pred:Spam")
print(f"  Actual:Not-Spam     {cm[0,0]:>5}         {cm[0,1]:>5}")
print(f"  Actual:Spam         {cm[1,0]:>5}         {cm[1,1]:>5}")
print(f"\n  TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")


In [ ]:
feature_names = list(spambase.data.features.columns)

# Discriminative score: Fisher-like ratio
#   score_i = (μ_spam - μ_not_spam)² / ((σ²_spam + σ²_not_spam)/2 + ε)
# Higher score → feature distributions differ more between classes.
spam_mask     = (y == 1)
not_spam_mask = (y == 0)

mu_spam     = X[spam_mask].mean(axis=0)
mu_not_spam = X[not_spam_mask].mean(axis=0)
var_spam     = X[spam_mask].var(axis=0)
var_not_spam = X[not_spam_mask].var(axis=0)

disc_scores   = (mu_spam - mu_not_spam) ** 2 / ((var_spam + var_not_spam) / 2 + 1e-9)
top_5_indices = np.argsort(disc_scores)[::-1][:5]

print("Top 5 most discriminative features (Fisher discriminant ratio):")
print(f"{'Rank':<6} {'Feature':<35} {'Score':>10} {'μ_spam':>10} {'μ_not_spam':>12}")
print("-" * 77)
for rank, idx in enumerate(top_5_indices, 1):
    print(f"{rank:<6} {feature_names[idx]:<35} {disc_scores[idx]:>10.2f} "
          f"{mu_spam[idx]:>10.4f} {mu_not_spam[idx]:>12.4f}")


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, idx in zip(axes, top_5_indices):
    spam_vals     = X[y == 1, idx]
    not_spam_vals = X[y == 0, idx]

    # Clip to 99th percentile to handle heavy tails
    clip_hi       = np.percentile(X[:, idx], 99)
    spam_vals     = spam_vals[spam_vals     <= clip_hi]
    not_spam_vals = not_spam_vals[not_spam_vals <= clip_hi]

    ax.hist(not_spam_vals, bins=30, alpha=0.6, label='Not Spam', density=True, color='steelblue')
    ax.hist(spam_vals,     bins=30, alpha=0.6, label='Spam',     density=True, color='tomato')
    ax.set_title(feature_names[idx], fontsize=9)
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Spam vs Not Spam (Top 5 Discriminative)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


## Summary and Discussion

### Results Table
*(Actual metric values are printed by the code cells above)*

| Metric | Value |
|--------|-------|
| Accuracy | (see output) |
| Precision | (see output) |
| Recall | (see output) |
| F1-Score | (see output) |

### Top 5 Discriminative Features
*(See code output — typically: `char_freq_$`, `word_freq_remove`, `word_freq_free`, `char_freq_!`, `word_freq_your`)*

### Discussion

**1. Why is Naive Bayes effective for spam detection despite the independence assumption?**
- Even though email features are correlated (e.g., "free" and "offer" co-occur in spam), the model still builds a correct decision boundary in many cases. The independence assumption leads to miscalibrated probability estimates, but the *argmax* class label is often still correct — "good enough for government work."
- Spam detection has a high signal-to-noise ratio: certain words and characters (like `$`, `!`, "free", "remove") are strongly associated with spam. Even a simple model captures this.
- With 57 features and 4601 training examples, the Gaussian parameters are estimated reliably from data.

**2. What are the limitations of this implementation?**
- **Gaussian assumption**: Word frequency features are highly skewed (many zeros, heavy right tail) — a Gaussian is a poor fit. A log-normal or Bernoulli/Multinomial distribution would be more appropriate.
- **Zero-variance features**: Handled by `var_smoothing`, but this is a heuristic.
- **Feature independence**: Many email features are correlated; the NB assumption degrades probability estimates.
- **No feature selection**: All 57 features contribute equally; some may add noise.

**3. How could you improve the classifier?**
- Apply `log(x + 1)` transform to frequency features before fitting Gaussian NB.
- Use Bernoulli NB (binarize features) or Multinomial NB for word-count features.
- Add feature selection (e.g., keep top-k by discriminative score) to reduce noise.

**4. What did you learn?**
- Naive Bayes is surprisingly effective and fast to train, even with strong modeling assumptions.
- Log-probabilities are critical — multiplying many small probabilities together causes numerical underflow without them.
- The most discriminative features match intuition: `$`, `!`, "free", and "remove" are classic spam signals.
